# From Training to Predictions

Notebook 1 selected and evaluated a Titanic classifier. Now turn that fitted pipeline into a reproducible prediction workflow: load it in a fresh kernel, check incoming records, make predictions, and explain the limits of the output.

Run notebook 1 through its final **Export for Notebook 3** cell first, then open this notebook with a fresh kernel. Notebook 2 practices training on a different dataset; this notebook returns to Titanic and needs no database connection. Keep the selected model and its preprocessing fixed throughout.

The new passenger records are **invented examples**, not new labeled observations. They demonstrate how the interface behaves; they cannot measure accuracy or validate real-world use.

## Learning Objectives

- Save and reload the complete fitted pipeline and its metadata.
- Distinguish training from inference: applying an existing model to new inputs.
- Read predicted classes and probabilities without treating them as guarantees.
- Separate missing values, invalid values, and values unfamiliar to the model.
- Process a batch while retaining rejected rows and their reasons.
- Investigate sensitivity to input changes and prepare a short model handover.

Work through the examples, complete the six exercises, then use the completion checklist at the end. The examples are runnable; your deliverable also needs your own records, an additional validation rule, and your interpretations.

## Prediction Workflow

<img src="assets/prediction_workflow.svg" width="800" style="max-width: 100%;" alt="New records are validated; accepted records pass through the saved preprocessing and model; predictions and rejected rows are included in the exported batch report.">

Training learned the transformations and the classifier. Inference reuses both: there is no new `fit` here. Invalid input is an interface problem even when a model would return a number for it.

## 1. Load the Saved Pipeline

In [ ]:
import json
from importlib.metadata import version
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
MODEL_PATH = ARTIFACT_DIR / "titanic_pipeline.joblib"
INFO_PATH = ARTIFACT_DIR / "titanic_model_info.json"
if not MODEL_PATH.is_file() or not INFO_PATH.is_file():
    raise FileNotFoundError(
        "Run notebook 1 through Export for Notebook 3, then reopen notebook 3 "
        "from the project root."
    )

The JSON file records the features, learned categories, package versions and evaluation already performed in notebook 1. Check the package versions before loading the model. Use the same project environment (`uv sync`) that produced the artifact.

Load only an artifact you created or trust: `joblib.load` can execute Python code. KNN also retains training observations in the fitted model, so a saved model is not necessarily free of training data. These artifacts stay in the Git-ignored `artifacts/` directory.

[Reference: scikit-learn model persistence](https://scikit-learn.org/stable/model_persistence.html).

In [ ]:
model_info = json.loads(INFO_PATH.read_text())
for package, saved_version in model_info["package_versions"].items():
    current_version = version(package)
    if current_version != saved_version:
        raise RuntimeError(
            f"{package}: artifact uses {saved_version}, environment uses "
            f"{current_version}. Recreate the artifact with the project environment."
        )

model = joblib.load(MODEL_PATH)
FEATURES = model_info["input_features"]
NUMERIC = model_info["numeric_features"]
CATEGORICAL = model_info["categorical_features"]
KNOWN_CATEGORIES = model_info["known_categories"]

pd.Series(
    {
        "Selected search": model_info["selected_search"],
        "Classifier": type(model.steps[-1][1]).__name__,
        "Features": ", ".join(FEATURES),
        "Recorded CV accuracy": model_info["evaluation"]["cv_accuracy"],
        "Recorded test accuracy": model_info["evaluation"]["test_accuracy"],
    },
    name="Saved model",
)

### Check: What Was Saved?

Explain why saving only the KNN classifier would not be sufficient for the six raw features. Which information would you need to reproduce the numerical and categorical transformations?

<details>
<summary>Show explanation</summary>

The classifier expects transformed features. The complete pipeline also retains the fitted imputers, scaler, category vocabulary and column routing. Recalculating those on the prediction batch would change the representation. The two accuracy values above document earlier evaluation; they are not scores for today's invented records.
</details>

## 2. Predict One New Record

`Survived = 1` is the positive class in this historical dataset. `predict` returns a class; `predict_proba` returns one column per class in `model.classes_`. Find the column for class 1 explicitly rather than relying on an unexplained column index.

In [ ]:
new_passenger = pd.DataFrame(
    [{"Age": 28, "SibSp": 0, "Parch": 0, "Fare": 7.9, "Sex": "male", "Embarked": "S"}]
)
positive_column = list(model.classes_).index(1)
single_prediction = model.predict(new_passenger)
single_probability = model.predict_proba(new_passenger)[:, positive_column]
pd.DataFrame(
    {"predicted_class": single_prediction, "probability_class_1": single_probability}
)

KNN probabilities depend on the class votes of nearby training observations (and any distance weighting). They are not automatically calibrated probabilities. With uniform weights they can change in discrete steps. A value near 0.5 means the classes receive similar votes; it does not tell you whether the input is valid or representative.

For this binary KNN, an exact probability tie selects the first class in `classes_`; manually using `probability >= 0.5` could therefore disagree with `predict`. We retain the fitted model's prediction rule.

[Reference: KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html).

### Exercise 1 - Your Own Prediction Requests

Create at least four invented records with unique `request_id` values: one ordinary record, one with a missing measurement, one with a category absent from training, and one with an impossible value. Use the six expected features and predict which records should be accepted, flagged for review, or rejected.

Keep these records for Exercise 6. Only use `model.predict` directly for the ordinary record for now; the next section adds validation for the other cases. Explain why you cannot compute their accuracy without true labels.

In [ ]:
# Create your request DataFrame and record your expected decisions.
# Select FEATURES before predicting: request_id is metadata, not a model input.

## 3. Define an Input Contract

An input contract specifies what the prediction workflow accepts. The following rules are conventions for this exercise; they do not prove that the historical model is suitable for new populations.

| Input | Contract |
|---|---|
| `request_id` | Present and unique within the batch; used to match results to requests. |
| Six feature columns | All columns must exist, even if some values are missing. Extra columns are rejected to catch schema mistakes. |
| `Age` | Number between 0 and 120, or missing. |
| `Fare` | Finite, nonnegative number, or missing; same units as the training CSV. |
| `SibSp`, `Parch` | Finite, nonnegative whole-number counts, or missing. |
| `Sex`, `Embarked` | Categories in the training vocabulary are familiar. Unfamiliar categories are flagged for review. Missing values use the fitted imputer. |

Missing values are not the same as missing columns. The imputer can fill an absent measurement, but cannot reconstruct the meaning of an absent feature column. A value outside a training range can be physically valid while being unfamiliar to the model.

In [ ]:
contract = pd.DataFrame(
    {
        "feature": FEATURES,
        "kind": ["numeric" if name in NUMERIC else "categorical" for name in FEATURES],
        "training_categories": [KNOWN_CATEGORIES.get(name, []) for name in FEATURES],
    }
)
contract

### Exercise 2 - Triage Before Predicting

Load the supplied batch below. Before running the validator, write a small decision table for its records: `predicted`, `review`, or `rejected`, with a reason. Pay particular attention to a missing age, age 100, age -5, a nonnumeric age, a fractional sibling count, an unknown port, and a row with no feature values.

Decide whether a prediction should be displayed for a review case, and explain how you would distinguish it from an ordinary prediction.

In [ ]:
batch = pd.read_csv(PROJECT_ROOT / "data" / "titanic_prediction_requests.csv")
batch

In [ ]:
# Write your decision table before continuing.
# Suggested columns: request_id, expected_status, explanation.

## 4. Validate Without Refitting

Start with the batch structure. A missing column or ambiguous request identifier blocks the whole batch because we cannot reliably assemble inputs and match outputs. Value errors are handled row by row below, so one bad age need not prevent other records from being processed.

In [ ]:
def check_structure(records):
    """Check the schema and identifiers before inspecting feature values."""
    expected = {"request_id", *FEATURES}
    if records.empty:
        raise ValueError("The batch must contain at least one request.")
    if not records.columns.is_unique:
        raise ValueError("Column names must be unique.")
    missing = expected - set(records.columns)
    extra = set(records.columns) - expected
    if missing or extra:
        raise ValueError(
            f"Missing columns: {sorted(missing)}; extra columns: {sorted(extra)}"
        )
    ids = records["request_id"]
    if (
        ids.isna().any()
        or ids.astype(str).str.strip().eq("").any()
        or ids.duplicated().any()
    ):
        raise ValueError("request_id must be present and unique within the batch.")


check_structure(batch)
print("Batch structure accepted.")

In [ ]:
# A missing column is a schema error, not a measurement for the imputer to fill.
try:
    check_structure(batch.drop(columns="Age"))
except ValueError as error:
    print("Expected rejection:", error)

Convert numerical inputs explicitly, but retain a record of every conversion failure. `errors="coerce"` below is useful only because failed conversions are subsequently rejected; treating their resulting NaN as an ordinary missing measurement would conceal bad input.

In [ ]:
def prepare_inputs(records):
    """Normalize numerical types without learning new preprocessing statistics."""
    prepared = records.copy()
    prepared[NUMERIC] = records[NUMERIC].apply(pd.to_numeric, errors="coerce")
    return prepared


prepared_batch = prepare_inputs(batch)
prepared_batch[NUMERIC].dtypes

In [ ]:
def find_issues(raw, prepared):
    """Separate invalid measurements from valid but unfamiliar inputs."""
    issues = []

    def add(mask, field, severity, reason):
        for request_id in raw.loc[mask, "request_id"]:
            issues.append((request_id, field, severity, reason))

    for column in NUMERIC:
        values = prepared[column]
        add(
            raw[column].notna() & values.isna(),
            column,
            "reject",
            f"{column}: not a number",
        )
        add(
            values.notna() & ~np.isfinite(values),
            column,
            "reject",
            f"{column}: not finite",
        )
        add(values.lt(0), column, "reject", f"{column}: negative value")
        bounds = model_info["numeric_training_ranges"][column]
        outside = values.lt(bounds["min"]) | values.gt(bounds["max"])
        add(outside, column, "review", f"{column}: outside the observed training range")
    add(
        prepared["Age"].gt(120),
        "Age",
        "reject",
        "Age: exceeds the exercise limit of 120",
    )
    for column in ["SibSp", "Parch"]:
        values = prepared[column]
        fractional = np.isfinite(values) & values.where(np.isfinite(values)).mod(1).ne(
            0
        )
        add(fractional, column, "reject", f"{column}: a count must be a whole number")
    for column in CATEGORICAL:
        unknown = raw[column].notna() & ~raw[column].isin(KNOWN_CATEGORIES[column])
        add(unknown, column, "review", f"{column}: category absent from training")
    add(
        raw[FEATURES].isna().all(axis=1),
        "all",
        "review",
        "All feature values are missing",
    )
    return pd.DataFrame(issues, columns=["request_id", "field", "severity", "reason"])


issues = find_issues(batch, prepared_batch)
issues

The encoder uses `handle_unknown="ignore"`: an unfamiliar category becomes an all-zero block for that categorical feature. It does not learn a new category or prove that the prediction is reliable. We preserve a review flag even though the pipeline can return an output.

Rejected rows may also have review reasons; rejection takes priority. The range check uses training statistics from the saved metadata, not new batch statistics.

[Reference: OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html).

### Exercise 3 - Add a Rule and Investigate a Schema Failure

Extend the validator with a rule of your choice. For example, flag rows missing at least three of the six features, or flag blank/whitespace category strings instead of treating them as new category names. Keep the existing checks.

Create at least two extra cases to demonstrate your rule, and explain whether it should reject or flag each case. Also demonstrate the structural rejection of a duplicated `request_id` or an unexpected `Survived` column. These checks verify the input workflow; they are not new model-performance evaluations.

<details>
<summary>Hint for a missing-feature rule</summary>

Use `raw[FEATURES].isna().sum(axis=1)` to count absent measurements. Add a review reason for your chosen threshold, then inspect overlaps with existing reasons. The threshold is your explicit interface policy, not a learned model parameter.
</details>

In [ ]:
# Extend find_issues, add your demonstration cases, and explain the policy.
# Re-run the batch-processing cells after changing the validation rule.

## 5. Predict a Batch and Keep Every Request

Prediction uses only rows without rejection reasons. Review cases retain their predictions for inspection, with their status and reasons clearly visible. Rejected rows retain their identifiers but have no prediction.

The output must keep one row for every request, in the original order. Never silently drop difficult records and present the rest as the full batch.

In [ ]:
def predict_batch(records):
    """Validate requests and return predictions with explicit row-level status."""
    check_structure(records)
    # Request IDs preserve identity; reset pandas indices before assigning outputs.
    records = records.reset_index(drop=True)
    prepared = prepare_inputs(records)
    found_issues = find_issues(records, prepared)
    rejected_ids = found_issues.loc[found_issues["severity"].eq("reject"), "request_id"]
    review_ids = found_issues.loc[found_issues["severity"].eq("review"), "request_id"]

    result = records[["request_id"]].copy()
    result["status"] = "predicted"
    result.loc[result["request_id"].isin(review_ids), "status"] = "review"
    result.loc[result["request_id"].isin(rejected_ids), "status"] = "rejected"
    result["predicted_class"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    result["probability_class_1"] = np.nan
    accepted = ~result["status"].eq("rejected")
    if accepted.any():
        inputs = prepared.loc[accepted, FEATURES]
        result.loc[accepted, "predicted_class"] = model.predict(inputs)
        result.loc[accepted, "probability_class_1"] = model.predict_proba(inputs)[
            :, positive_column
        ]
    reasons = found_issues.groupby("request_id")["reason"].agg("; ".join)
    result["notes"] = result["request_id"].map(reasons).fillna("")
    return result, found_issues


results, issues = predict_batch(batch)
results

In [ ]:
status_counts = (
    results["status"]
    .value_counts()
    .reindex(["predicted", "review", "rejected"], fill_value=0)
)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(
    status_counts.index, status_counts.values, color=["#398454", "#c18a25", "#b44545"]
)
ax.set(title="Input handling for the invented batch", ylabel="Requests")
fig.tight_layout()

### Exercise 4 - Report Input Quality

Build a compact summary with the total requests, each status count, the percentage of requests with at least one missing feature, and the most frequent issue reasons. Count affected requests rather than issue entries: a single row can have several issues.

Compare your results with your triage table from Exercise 2 and explain any disagreement. State one additional check this workflow cannot perform from these columns alone, such as verifying the units or the source of a plausible-looking value.

In [ ]:
# Build your batch summary and compare it with your earlier decisions.

## 6. Read Probabilities Together with Input Status

A prediction with probability 0.9 can still come from an unfamiliar category or an input outside the training range. The probability describes the fitted classifier's output; the status describes input quality under our contract. They answer different questions.

In [ ]:
scored = results.loc[results["status"].ne("rejected")]
fig, ax = plt.subplots(figsize=(8, 4))
colors = scored["status"].map({"predicted": "#398454", "review": "#c18a25"})
ax.barh(scored["request_id"], scored["probability_class_1"], color=colors)
ax.axvline(0.5, color="gray", linestyle="--", label="Equal class probability")
ax.set(
    xlim=(0, 1),
    xlabel="Estimated P(Survived = 1)",
    title="Invented requests: probability and input status",
)
ax.legend()
fig.tight_layout()

Discuss: can the row with every feature missing still receive a prediction? Why? What evidence would you need before describing the displayed probabilities as well calibrated?

<details>
<summary>Show explanation</summary>

The fitted imputers replace the missing values and the classifier predicts on that constructed representation. It has not acquired information about this new passenger. Calibration concerns agreement between probabilities and observed outcomes across labeled examples; this unlabeled batch cannot establish it. Keep the review status and the absence of performance evidence visible.
</details>

## 7. Investigate Sensitivity Without Retraining

Hold five features fixed and vary age within a chosen range. This shows how the frozen model responds to one input change. It is not a causal experiment: changing age in this constructed record does not prove that age causes a change in survival.

In [ ]:
ages = np.arange(5, 81, 5)
sensitivity = pd.concat([new_passenger] * len(ages), ignore_index=True)
sensitivity["Age"] = ages
sensitivity_probability = model.predict_proba(sensitivity)[:, positive_column]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ages, sensitivity_probability, marker="o", color="#4076a6")
ax.set(
    xlabel="Age in the constructed record",
    ylabel="Estimated P(Survived = 1)",
    ylim=(0, 1),
)
fig.tight_layout()

### Exercise 5 - Compare Two Profiles

Choose two invented profiles and vary the same numerical feature for both, with all other inputs fixed within each profile. Plot both probability curves. You can investigate age or fare; document the range and identify any values outside the saved training range.

Describe the differences, any abrupt changes, and one conclusion the plot does **not** justify. Explain why tuning a model or threshold to make these invented examples look nicer would not establish better performance.

In [ ]:
# Create two profiles, vary one feature, plot both curves, and interpret them.

## 8. Export Results and Prepare a Handover

Export the complete result table, including rejected requests. A separate issue table preserves the reason-level detail. The model metadata documents earlier evaluation; the batch report documents processing today. Do not invent accuracy for this unlabeled batch.

In [ ]:
results.to_csv(ARTIFACT_DIR / "titanic_predictions.csv", index=False)
issues.to_csv(ARTIFACT_DIR / "titanic_input_issues.csv", index=False)
batch_report = {
    "input_file": "data/titanic_prediction_requests.csv",
    "model_file": MODEL_PATH.name,
    "request_count": len(batch),
    "status_counts": {name: int(count) for name, count in status_counts.items()},
    "requests_with_missing_features": int(batch[FEATURES].isna().any(axis=1).sum()),
    "ground_truth_available": False,
}
(ARTIFACT_DIR / "titanic_batch_report.json").write_text(
    json.dumps(batch_report, indent=2) + "\n"
)
pd.Series(batch_report, name="Batch report")

### Exercise 6 - Deliver Your Own Batch and Handover

Process the records you created in Exercises 1 and 3 using your extended validator. Export their predictions, issues and a report with their actual counts and source description. Save a short `artifacts/model_handover.md` containing:

1. The purpose and historical target of the model.
2. The six raw inputs and how missing, invalid and unknown values are handled.
3. Which recorded scores came from CV and from the final test.
4. Your additional validation rule and the reason for its severity.
5. Two limitations, including the lack of true labels for your new examples.
6. Your recommendation for handling review cases and what evidence is needed for a real use case.

Finally, restart the kernel and run this notebook again to check that the saved pipeline loads and the supplied examples work without notebook 1's in-memory variables. Re-run your own exercise cells as well. Confirm that the same ordinary records receive the same predictions before and after saving/loading.

In [ ]:
# Process your own batch, export its results and report, and write the handover.
# Use filenames distinct from the supplied example outputs.

## Completion Checklist

- The complete pipeline loads from disk in a fresh kernel; no training happens here.
- Your own examples include ordinary, missing, unfamiliar and invalid inputs.
- Your additional validation rule is explained and demonstrated.
- Every request has one output row; rejected requests have no prediction.
- Your batch summary counts requests correctly and distinguishes input quality from model performance.
- Your two-profile plot is interpreted without a causal claim.
- Exported files describe your actual batch, and the handover explains limitations.

After completing the exercises, compare your choices with the [worked solution](solutions/3_from_training_to_predictions.ipynb). It includes one possible validation policy, batch summary, sensitivity comparison, and handover.

### Optional Extensions

- Turn your validated batch workflow into a small `predict.py` script with input and output paths. Keep the saved model fixed.
- Compare whole-batch and chunked CSV prediction; check output order and identifier uniqueness across chunks.
- Exchange an invented input batch with another student and explain every rejection or review flag.